In [1]:
import pandas as pd
import numpy as np

from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Labs"
)

xpt_file = DATA_DIR / "HDL_G.xpt"
csv_file = DATA_DIR / "HDL_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the original pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2011-2012 HDL_G XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBDHDD",
    "LBDHDDSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HDL_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 7821:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 7,821 rows x 3 columns."
)


# ============================================================
# 4. DEFINE VARIABLE TYPES
# ============================================================

#
# SEQN
#     Respondent identifier
#     whole-number
#
# LBDHDD
#     Direct HDL cholesterol in mg/dL
#     released as whole-number values
#
# LBDHDDSI
#     Direct HDL cholesterol in mmol/L
#     genuine decimal laboratory value
#
# IMPORTANT:
#
# Do NOT convert LBDHDDSI to Int64.
# Do NOT round it merely to remove .0.
#

INTEGER_COLUMNS = [
    "SEQN",
    "LBDHDD"
]


DECIMAL_COLUMNS = [
    "LBDHDDSI"
]


print("\n--- VARIABLE TYPE PLAN ---")

print(
    "Whole-number variables:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas artifact:
#
#     5.397605346934028e-79
#
# Attached HDL_G.xpt:
#
#     tiny values = 0
#
# Therefore NO correction is needed.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros:",
    true_zero_total
)

print(
    "Tiny XPORT values:",
    tiny_total
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT values "
        "found in HDL_G. Do not automatically "
        "replace them with zero."
    )


if true_zero_total != 0:

    raise ValueError(
        "STOP: Unexpected true numeric zeros "
        "found in HDL_G."
    )


print(
    "PASS: No tiny XPORT artifacts."
)

print(
    "PASS: No numeric zeros."
)


# ============================================================
# 6. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# This check MUST occur before any Int64 conversion.
#
# Expected:
#
#     SEQN       whole-number
#     LBDHDD     whole-number
#     LBDHDDSI   genuine decimal
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns != ["LBDHDDSI"]:

    raise ValueError(
        "Unexpected decimal-variable pattern.\n"
        "Expected: ['LBDHDDSI']\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "PASS: LBDHDDSI is the only genuine "
    "decimal variable."
)


# ============================================================
# 7. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. FORMAT LBDHDD WITHOUT .0
# ============================================================

#
# LBDHDD is HDL cholesterol in mg/dL.
#
# The released values are whole-number values.
#
# Example:
#
#     41.0 -> 41
#
# NO .round() is used.
#

values = (
    df["LBDHDD"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.all(
    np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "LBDHDD unexpectedly contains "
        "fractional values."
    )


df["LBDHDD"] = (
    df["LBDHDD"]
    .astype("Int64")
)


print(
    "PASS: LBDHDD converted to Int64 "
    "without rounding."
)


# ============================================================
# 9. PRESERVE LBDHDDSI EXACTLY
# ============================================================

#
# LBDHDDSI is a genuine decimal laboratory variable.
#
# DO NOT:
#
#     round()
#     astype(Int64)
#     use float_format
#
# It must remain unchanged from XPT.
#

if not np.allclose(
    source_df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    ),
    df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    ),
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBDHDDSI changed unexpectedly."
    )


print(
    "PASS: LBDHDDSI preserved exactly."
)


# ============================================================
# 10. CDC CHECK - LBDHDD
# ============================================================

#
# CDC 2011-2012:
#
# LBDHDD
#
# Direct HDL-Cholesterol (mg/dL)
#
# Range:
#
#     14 - 175
#
# Count:
#
#     6,989
#
# Missing:
#
#     832
#

s = df["LBDHDD"]


print("\n--- LBDHDD ---")


print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    s.min() == 14
    and
    s.max() == 175
    and
    int(
        s.notna()
        .sum()
    ) == 6989
    and
    int(
        s.isna()
        .sum()
    ) == 832
):

    raise ValueError(
        "LBDHDD differs from CDC."
    )


print(
    "PASS: LBDHDD matches CDC."
)


# ============================================================
# 11. CDC CHECK - LBDHDDSI
# ============================================================

#
# CDC 2011-2012:
#
# LBDHDDSI
#
# Direct HDL-Cholesterol (mmol/L)
#
# Range:
#
#     0.36 - 4.53
#
# Count:
#
#     6,989
#
# Missing:
#
#     832
#

s = df["LBDHDDSI"]


print("\n--- LBDHDDSI ---")


print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    np.isclose(
        s.min(),
        0.36,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        4.53,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    ) == 6989
    and
    int(
        s.isna()
        .sum()
    ) == 832
):

    raise ValueError(
        "LBDHDDSI differs from CDC."
    )


print(
    "PASS: LBDHDDSI matches CDC."
)


# ============================================================
# 12. VERIFY CDC SI CONVERSION
# ============================================================

#
# CDC states:
#
#     HDL mmol/L
#
#       =
#
#     HDL mg/dL × 0.02586
#
#
# Released LBDHDDSI values are represented to
# two decimal places.
#
# Use Decimal + ROUND_HALF_UP rather than Python round()
# because Python uses banker's rounding at .5 boundaries.
#

def convert_hdl_to_si(
    mg_dl
):

    converted = (
        Decimal(
            str(int(mg_dl))
        )
        *
        Decimal("0.02586")
    )


    return float(
        converted.quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )
    )


paired = (
    df[
        [
            "LBDHDD",
            "LBDHDDSI"
        ]
    ]
    .dropna()
)


expected_si = (
    paired["LBDHDD"]
    .apply(
        convert_hdl_to_si
    )
    .to_numpy(
        dtype=float
    )
)


actual_si = (
    paired["LBDHDDSI"]
    .to_numpy(
        dtype=float
    )
)


si_mismatches = int(
    np.sum(
        expected_si
        !=
        actual_si
    )
)


print("\n--- CDC SI CONVERSION CHECK ---")


print(
    "Pairs checked:",
    f"{len(paired):,}"
)


print(
    "Conversion mismatches:",
    si_mismatches
)


if si_mismatches != 0:

    raise ValueError(
        "LBDHDDSI does not match "
        "LBDHDD × 0.02586."
    )


print(
    "PASS: All 6,989 LBDHDDSI values "
    "match CDC conversion exactly."
)


# ============================================================
# 13. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# HDL_G requires NO numeric correction.
#
# Therefore the numeric values must remain exactly
# equal to the original source XPT.
#
# Representation changes only:
#
#     SEQN
#     62161.0 -> 62161
#
#     LBDHDD
#     41.0 -> 41
#
# LBDHDDSI remains decimal.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 3 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 14. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO:
#
#     float_format=
#
# NO:
#
#     rounding
#
# This preserves the genuine LBDHDDSI decimal values.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 15. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 16. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 3 CSV variables round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 17. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 18. READ RAW CSV AS TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 19. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 20. .0 FORMAT CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# DO NOT search LBDHDDSI for ".0" errors.
#
# It is a genuine decimal measurement.
#
# Only:
#
#     SEQN
#     LBDHDD
#
# should be checked for unnecessary .0.
#

print("\n--- INTEGER .0 FORMAT CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 in "
    "SEQN or LBDHDD."
)


# ============================================================
# 21. EXACT DECIMAL FIDELITY
# ============================================================

expected_si = (
    source_df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    )
)


exported_si = (
    roundtrip_df["LBDHDDSI"]
    .to_numpy(
        dtype=float
    )
)


if not np.allclose(
    expected_si,
    exported_si,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBDHDDSI decimal fidelity failed."
    )


print(
    "PASS: LBDHDDSI decimal values "
    "preserved exactly."
)


# ============================================================
# 22. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 23. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "HDL_G CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Whole-number variables:",
    INTEGER_COLUMNS
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "LBDHDD valid results:",
    int(
        df["LBDHDD"]
        .notna()
        .sum()
    )
)


print(
    "LBDHDDSI valid results:",
    int(
        df["LBDHDDSI"]
        .notna()
        .sum()
    )
)


print(
    "SI conversion mismatches:",
    si_mismatches
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0 in integer fields: "
    "0 expected"
)


print(
    "Lab decimal values altered: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 HDL_G XPT -> CSV
Rows: 7,821
Columns: 3
PASS: Structure = 7,821 rows x 3 columns.

--- VARIABLE TYPE PLAN ---
Whole-number variables: ['SEQN', 'LBDHDD']
Genuine decimal variables: ['LBDHDDSI']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny XPORT values: 0
PASS: No tiny XPORT artifacts.
PASS: No numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
LBDHDD: 0 fractional observations
LBDHDDSI: 6,989 fractional observations
PASS: LBDHDDSI is the only genuine decimal variable.

--- SEQN ---
Minimum: 62161
Maximum: 71916
Missing: 0
Duplicates: 0
PASS: SEQN validated.
PASS: LBDHDD converted to Int64 without rounding.
PASS: LBDHDDSI preserved exactly.

--- LBDHDD ---
Minimum: 14
Maximum: 175
Non-missing: 6989
Missing: 832
PASS: LBDHDD matches CDC.

--- LBDHDDSI ---
Minimum: 0.36
Maximum: 4.53
Non-missing: 6989
Missing: 832
PASS: LBDHDDSI matches CDC.

--- CDC SI CONVERSION CHECK ---
Pairs checked: 6,989
Conversion mismatches: 0
PASS: A